# RGB Training v4 — CNN Residual dan SVM Standalone

Notebook ini melatih **CNN RGB baru** dengan empat tahap residual, kemudian menggunakan checkpoint terbaik sebagai feature extractor beku untuk melatih **SVM standalone RGB**. Tahap SVM menggunakan `GridSearchCV`, `StandardScaler`, `Normalizer`, dan `SVC` dengan kernel RBF. Model AVG, NTSC, dan artefak v1/v2/v3 tidak diubah.

Split tetap **40.000 training / 10.000 validation** (seed `30092026`). Official test **tidak dibaca** di sini. Untuk mengurangi perhitungan ulang, fitur RGB original (A) dan rata-rata horizontal flip (B) disimpan dengan format yang kompatibel dengan Feature Engineering Combination v4. Model SVM standalone memakai fitur **B**; SVM fusion dilatih terpisah. Semua keluaran tersimpan di `outputs/cifar10_rgb_residual_v4/rgb/`.

**Catatan:** Validation juga digunakan dalam pemilihan checkpoint CNN sehingga skor SVM development bukan estimasi yang sepenuhnya independen dari proses pemilihan CNN. Evaluasi official test dilaksanakan hanya dalam RGB Testing v4.

## 1. Konfigurasi lingkungan dan direktori

Gunakan kernel `envGPU` dari direktori `tugas-7`. Arsitektur dan parameter CNN tidak diubah dari versi v4 sebelumnya. Jika checkpoint CNN dan laporan pelatihan yang cocok **sudah ada**, notebook menggunakannya kembali tanpa retraining; artefak yang tidak lengkap atau memiliki hash berbeda akan ditolak. `BATCH_SIZE=32` adalah titik awal konservatif untuk RTX 3050 6 GB.

In [1]:
from pathlib import Path
import os, gc, json, hashlib, time, shutil
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

SEED = 30092026
RUN_ID = "cifar10_rgb_residual_v4"
BATCH_SIZE = 32
MAX_EPOCHS = 120
LEARNING_RATE = 0.0005
DROPOUT_RATE = 0.35
WEIGHT_DECAY = 0.0001
ROOT = Path.cwd().resolve()
if not (ROOT / "outputs").is_dir() and (ROOT / "tugas-7" / "outputs").is_dir():
    ROOT = ROOT / "tugas-7"
RGB_OUT = ROOT / "outputs" / RUN_ID / "rgb"
for part in ("model", "fitur", "laporan"):
    (RGB_OUT / part).mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
if tf.config.list_physical_devices("GPU"):
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
print('Akar:', ROOT)
print('Checkpoint baru:', RGB_OUT / 'model' / 'cnn.keras')
print('GPU:', tf.config.list_physical_devices('GPU'))

I0000 00:00:1791499765.991141  231151 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791499766.030286  231151 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791499766.981524  231151 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Akar: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts
Checkpoint baru: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_rgb_residual_v4/rgb/model/cnn.keras
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Data resmi dan split yang kompatibel dengan AVG/NTSC

Hanya bagian training dari `cifar10.load_data()` yang digunakan. Indeks disortir dan di-hash agar dapat dicocokkan secara identik dengan arsip v1/v2. Label validation tidak digunakan dalam update bobot CNN, tetapi dipakai untuk memilih checkpoint.

In [2]:
from tensorflow.keras.datasets import cifar10
(x_all, y_all), _ = cifar10.load_data()
y_all = y_all.ravel().astype(np.int64)
assert x_all.shape == (50000, 32, 32, 3)
idx_train, idx_val = train_test_split(np.arange(50000),test_size=.2,random_state=SEED,stratify=y_all)
idx_train, idx_val = np.sort(idx_train), np.sort(idx_val)
split_hash = hashlib.sha256(idx_train.tobytes() + idx_val.tobytes()).hexdigest()
x_float = x_all.astype(np.float32) / np.float32(255.0)
assert len(idx_train)==40000 and len(idx_val)==10000 and len(np.intersect1d(idx_train,idx_val))==0
print('Training:',len(idx_train),'| Validation:',len(idx_val),'| Hash:',split_hash)

Training: 40000 | Validation: 10000 | Hash: 16004a1ac8a89ac92ba6280367a7fb7b284c3643abb1c9d10409d6739ef31305


## 3. Residual block dan arsitektur CNN empat tahap

Fitur SVM berasal **langsung** dari pooling 512 kanal, bukan dari Dense classifier. `ZeroPadding2D` diikuti `RandomCrop` digunakan untuk augmentasi crop selama training; saat inference `RandomCrop` memakai crop pusat. `RandomFlip`, `RandomRotation`, dan `RandomZoom` tidak mengubah input secara acak ketika `training=False`. Semua model ini custom CNN tanpa pretrained weights.

In [3]:
def residual_block(x, filters, stride=1, name='res'):
    shortcut = x
    x = tf.keras.layers.Conv2D(filters,3,strides=stride,padding='same',use_bias=False,name=f'{name}_conv1')(x)
    x = tf.keras.layers.BatchNormalization(name=f'{name}_bn1')(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(filters,3,padding='same',use_bias=False,name=f'{name}_conv2')(x)
    x = tf.keras.layers.BatchNormalization(name=f'{name}_bn2')(x)
    if stride != 1 or int(shortcut.shape[-1]) != filters:
        shortcut = tf.keras.layers.Conv2D(filters,1,strides=stride,padding='same',use_bias=False,name=f'{name}_skipconv')(shortcut)
        shortcut = tf.keras.layers.BatchNormalization(name=f'{name}_skipbn')(shortcut)
    return tf.keras.layers.Activation('relu',name=f'{name}_relu')(tf.keras.layers.Add()([x,shortcut]))

def make_rgb_cnn(dropout=DROPOUT_RATE):
    inputs = tf.keras.Input((32,32,3), name='rgb_input')
    x = tf.keras.layers.ZeroPadding2D(4,name='augmentation_padding')(inputs)
    x = tf.keras.layers.RandomCrop(32,32,seed=SEED,name='augmentation_crop')(x)
    x = tf.keras.layers.RandomFlip('horizontal',seed=SEED,name='augmentation_flip')(x)
    x = tf.keras.layers.RandomRotation(0.03,fill_mode='reflect',seed=SEED,name='augmentation_rotate')(x)
    x = tf.keras.layers.RandomZoom(0.08,fill_mode='reflect',seed=SEED,name='augmentation_zoom')(x)
    x = tf.keras.layers.Conv2D(64,3,padding='same',use_bias=False,name='stem_conv')(x)
    x = tf.keras.layers.BatchNormalization(name='stem_bn')(x)
    x = tf.keras.layers.Activation('relu',name='stem_relu')(x)
    for filters, stride in ((64,1),(128,2),(256,2),(512,1)):
        x = residual_block(x,filters,stride,name=f'stage{filters}_block1')
        x = residual_block(x,filters,1,name=f'stage{filters}_block2')
    features = tf.keras.layers.GlobalAveragePooling2D(name='svm_features',dtype='float32')(x)
    x = tf.keras.layers.Dropout(dropout,name='head_dropout')(features)
    x = tf.keras.layers.Dense(256,activation='relu',dtype='float32',name='head_dense')(x)
    outputs = tf.keras.layers.Dense(10,activation='softmax',dtype='float32',name='classification')(x)
    return tf.keras.Model(inputs,outputs,name='custom_rgb_residual_v4')

cnn = make_rgb_cnn()
assert cnn.get_layer('svm_features').output.shape[-1] == 512
print('Parameter CNN:',cnn.count_params())
cnn.summary()

I0000 00:00:1791499769.793207  231151 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4139 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 6GB Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Parameter CNN: 11312330


Model: "custom_rgb_residual_v4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ rgb_input           │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ augmentation_paddi… │ (None, 40, 40, 3) │          0 │ rgb_input[0][0]   │
│ (ZeroPadding2D)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ augmentation_crop   │ (None, 32, 32, 3) │          0 │ augmentation_pad… │
│ (RandomCrop)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ augmentation_flip   │ (None, 32, 32, 3) │          0 │ augmentation_cro… │
│ (RandomFlip)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ augmentation_rotate │ (None, 32, 32, 3) │          0 │ augmentation_fli… │
│ (RandomRotation)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ augmentation_zoom   │ (None, 32, 32, 3) │          0 │ augmentation_rot… │
│ (RandomZoom)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 32, 32,    │      1,728 │ augmentation_zoo… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 32, 32,    │        256 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_relu           │ (None, 32, 32,    │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage64_block1_con… │ (None, 32, 32,    │     36,864 │ stem_relu[0][0]   │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage64_block1_bn1  │ (None, 32, 32,    │        256 │ stage64_block1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 32, 32,    │          0 │ stage64_block1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage64_block1_con… │ (None, 32, 32,    │     36,864 │ activation[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage64_block1_bn2  │ (None, 32, 32,    │        256 │ stage64_block1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 32, 32,    │          0 │ stage64_block1_b… │
│                     │ 64)               │            │ stem_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage64_block1_relu │ (None, 32, 32,    │          0 │ add[0][0]         │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage64_block2_con… │ (None, 32, 32,    │     36,864 │ stage64_block1_r

 Total params: 11,312,330 (43.15 MB)

 Trainable params: 11,302,730 (43.12 MB)

 Non-trainable params: 9,600 (37.50 KB)

## 4. Compile dan callbacks training

AdamW dan cosine decay dipakai untuk stabilitas optimasi. Checkpoint disimpan berdasarkan `val_accuracy`. `EarlyStopping` mencegah training terus-menerus saat validation tidak membaik. Pelatihan dimulai dari bobot acak, bukan checkpoint lama.

In [4]:
steps = int(np.ceil(len(idx_train) / BATCH_SIZE))
schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LEARNING_RATE,
    decay_steps=MAX_EPOCHS * steps,
    alpha=0.03,
)
cnn.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=schedule, weight_decay=WEIGHT_DECAY),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy'],
)
checkpoint = RGB_OUT / 'model' / 'best_checkpoint.keras'
final_path = RGB_OUT / 'model' / 'cnn.keras'
report_path = RGB_OUT / 'laporan' / 'cnn_training.json'
REUSE_EXISTING_CNN = final_path.exists() and report_path.exists()
if REUSE_EXISTING_CNN:
    saved_report = json.loads(report_path.read_text(encoding='utf-8'))
    h = hashlib.sha256()
    with final_path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    if (saved_report.get('run_id') != RUN_ID
        or saved_report.get('split_hash') != split_hash
        or saved_report.get('cnn_sha256') != h.hexdigest()
        or saved_report.get('feature_dim') != 512):
        raise ValueError('Checkpoint CNN RGB v4 lama tidak cocok dengan run/split. Tidak dilakukan retraining otomatis.')
    print('Checkpoint CNN RGB v4 sudah tersedia dan cocok; training CNN akan dilewati.')
elif_final_missing = not final_path.exists() and not report_path.exists() and not checkpoint.exists()
if not REUSE_EXISTING_CNN and not elif_final_missing:
    raise FileExistsError('Artefak CNN v4 hanya sebagian tersedia. Periksa checkpoint dan laporan sebelum melanjutkan.')
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(str(checkpoint), monitor='val_accuracy', mode='max', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=18, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.TerminateOnNaN(),
]
print('LR:', LEARNING_RATE, '| Batch:', BATCH_SIZE, '| Maksimum epoch:', MAX_EPOCHS)

LR: 0.0005 | Batch: 32 | Maksimum epoch: 120


## 5. Pelatihan CNN RGB atau penggunaan checkpoint yang sudah ada

CNN dilatih dari bobot acak **hanya jika** checkpoint v4 belum tersedia. Jika checkpoint dan metadata telah diverifikasi, pelatihan CNN dilewati; tahapan SVM berikutnya tetap bisa berjalan. Jangan menjalankan training CNN lain secara paralel.

In [5]:
if REUSE_EXISTING_CNN:
    history = None
    elapsed = float(saved_report.get('training_seconds', 0.0))
    print('CNN RGB v4 tidak dilatih ulang. Best validation:', f"{saved_report['best_cnn_validation_accuracy']:.2%}")
else:
    start = time.perf_counter()
    history = cnn.fit(
        x_float[idx_train], y_all[idx_train],
        validation_data=(x_float[idx_val], y_all[idx_val]),
        batch_size=BATCH_SIZE, epochs=MAX_EPOCHS, callbacks=callbacks, verbose=1,
    )
    elapsed = time.perf_counter() - start
    print('Epoch dijalankan:', len(history.epoch), '| Durasi menit:', round(elapsed / 60, 1))

Epoch 1/120


E0000 00:00:1791499778.452522  231151 util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
I0000 00:00:1791499779.180827  231228 cuda_dnn.cc:461] Loaded cuDNN version 92600


1250/1250 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.4200 - loss: 1.5874
Epoch 1: val_accuracy improved from None to 0.55020, saving model to /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_rgb_residual_v4/rgb/model/best_checkpoint.keras

Epoch 1: finished saving model to /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_rgb_residual_v4/rgb/model/best_checkpoint.keras
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 107s 74ms/step - accuracy: 0.4200 - loss: 1.5874 - val_accuracy: 0.5502 - val_loss: 1.2746
Epoch 2/120
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.5929 - loss: 1.1315
Epoch 2: val_accuracy improved from 0.55020 to 0.56840, saving model to /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_rgb_residual_v4/rgb/model/best_checkpoint.keras

Epoch 2: finished saving model to /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_rgb_residual_v4/rgb/model/best_checkpoint.keras

## 6. Verifikasi checkpoint terbaik dan metadata CNN

Checkpoint terbaik disalin ke `cnn.keras` **hanya untuk training baru**. Apabila model sudah tersedia, file tidak ditimpa. Hash SHA-256, identitas split, dan nama feature layer digunakan oleh RGB Extraction, Combination, dan Testing.

In [6]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

if REUSE_EXISTING_CNN:
    report = saved_report
    assert sha256_file(final_path) == report['cnn_sha256']
else:
    if not checkpoint.is_file():
        raise FileNotFoundError('Checkpoint CNN RGB terbaik tidak ditemukan')
    shutil.copyfile(checkpoint, final_path)
    best_val = float(max(history.history['val_accuracy']))
    report = {
        'run_id': RUN_ID, 'source': 'custom CNN from random initialization',
        'seed': SEED, 'split_hash': split_hash, 'n_train': 40000, 'n_val': 10000,
        'architecture': '4 residual stages 64/128/256/512, 2 blocks per stage',
        'extract_layer': 'svm_features', 'feature_dim': 512,
        'input_normalization': 'float32 / 255', 'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE, 'weight_decay': WEIGHT_DECAY,
        'dropout': DROPOUT_RATE, 'max_epochs': MAX_EPOCHS,
        'epochs_run': len(history.epoch), 'training_seconds': elapsed,
        'best_cnn_validation_accuracy': best_val,
        'cnn_sha256': sha256_file(final_path),
        'history': {k: [float(z) for z in v] for k, v in history.history.items()},
    }
    if report_path.exists():
        raise FileExistsError('Laporan CNN sudah ada; tidak menimpa eksperimen sebelumnya')
    report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')

best = tf.keras.models.load_model(final_path, compile=False)
assert int(best.get_layer('svm_features').output.shape[-1]) == 512
print('CNN validation:', f"{report['best_cnn_validation_accuracy']:.2%}", '| SHA:', report['cnn_sha256'][:16])
print('CNN:', final_path)
del best, cnn
tf.keras.backend.clear_session()
gc.collect()

CNN validation: 92.96% | SHA: cea218a9171acf35
CNN: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_rgb_residual_v4/rgb/model/cnn.keras


0

## 7. Konfigurasi SVM standalone dan fitur yang digunakan

SVM RGB standalone menggunakan **512-D `svm_features`** dari checkpoint CNN v4, bukan output kelas Softmax. Representasi yang digunakan adalah **B: rata-rata fitur gambar original dan horizontal flip**, dengan CNN dibekukan (`training=False`). Ini tidak sama dengan SVM fusion tujuh fitur. Jika arsip RGB A/B sudah dibuat oleh RGB Extraction v4, arsip tersebut digunakan ulang setelah verifikasi; jika belum tersedia, tahap berikutnya membuatnya sehingga Extraction v4 boleh dilewati.

In [7]:
import joblib
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, Normalizer
from sklearn.svm import SVC

RGB_FEATURE_PATH = RGB_OUT / 'fitur' / 'train_features.npz'
GRID_REPORT_PATH = RGB_OUT / 'laporan' / 'svm_grid_search.json'
DEVELOPMENT_REPORT_PATH = RGB_OUT / 'laporan' / 'svm_development.json'
SVM_ARTIFACT_PATH = RGB_OUT / 'model' / 'svm.joblib'
SVM_REPORT_PATH = RGB_OUT / 'laporan' / 'svm_training.json'
SVM_REPRESENTATION = 'B'
SVM_EXTRACT_BATCH = 64
RUN_FINAL_SVM_50K = True
SVM_PARAM_GRID = {'svm__C': [1.0, 10.0], 'svm__gamma': ['scale', 0.5, 1.0]}
print('Classifier:', 'SVM RGB standalone', '| Fitur:', SVM_REPRESENTATION,
      '| Kandidat GridSearchCV:', 6)

Classifier: SVM RGB standalone | Fitur: B | Kandidat GridSearchCV: 6


## 8. Muat atau ekstrak fitur RGB A dan B pada 50.000 training images

Fitur A diambil dari gambar asli, sedangkan fitur B adalah rata-rata fitur original dan horizontal flip. Jika arsip sudah ada, cocokkan seed, split, label, dimensi, identitas CNN, dan sample order. Jika belum ada, jalankan inference pada satu CNN beku dan simpan arsip dengan skema yang dapat dipakai langsung oleh **Feature Engineering Combination v4**. Tidak ada pembacaan official test dan tidak ada pembaruan bobot CNN.

In [8]:
if RGB_FEATURE_PATH.is_file():
    with np.load(RGB_FEATURE_PATH, allow_pickle=False) as cached:
        if (str(cached['run_id']) != RUN_ID
            or str(cached['cnn_sha256']) != report['cnn_sha256']
            or str(cached['split_hash']) != split_hash
            or str(cached['layer_name']) != 'svm_features'
            or cached['features_A'].shape != (50000, 512)
            or cached['features_B'].shape != (50000, 512)
            or not np.array_equal(cached['train_idx'], idx_train)
            or not np.array_equal(cached['val_idx'], idx_val)
            or not np.array_equal(cached['indices'], np.arange(50000))
            or not np.array_equal(cached['y'], y_all)):
            raise ValueError('Arsip fitur RGB v4 tidak kompatibel dengan model atau split saat ini')
        X_A = cached['features_A'].astype(np.float32)
        X_B = cached['features_B'].astype(np.float32)
    print('Menggunakan arsip RGB A/B yang telah tersedia:', RGB_FEATURE_PATH)
else:
    tf.keras.backend.clear_session()
    frozen_cnn = tf.keras.models.load_model(final_path, compile=False)
    extractor = tf.keras.Model(frozen_cnn.input, frozen_cnn.get_layer('svm_features').output)
    X_A = np.empty((50000, 512), dtype=np.float32)
    X_B = np.empty((50000, 512), dtype=np.float32)
    started = time.perf_counter()
    for start in range(0, 50000, SVM_EXTRACT_BATCH):
        end = min(start + SVM_EXTRACT_BATCH, 50000)
        batch = x_float[start:end]
        original = extractor(batch, training=False).numpy().astype(np.float32)
        flipped = extractor(np.ascontiguousarray(batch[:, :, ::-1, :]), training=False).numpy().astype(np.float32)
        X_A[start:end] = original
        X_B[start:end] = (original + flipped) * np.float32(0.5)
        if end % 6400 == 0 or end == 50000:
            print(f'Ekstraksi RGB: {end}/50000', flush=True)
    del extractor, frozen_cnn
    tf.keras.backend.clear_session()
    gc.collect()
    assert np.isfinite(X_A).all() and np.isfinite(X_B).all()
    temp_path = RGB_FEATURE_PATH.with_suffix('.npz.tmp')
    with temp_path.open('wb') as output:
        np.savez_compressed(
            output, features_A=X_A, features_B=X_B, y=y_all,
            indices=np.arange(50000), train_idx=idx_train, val_idx=idx_val,
            run_id=RUN_ID, split_hash=split_hash,
            cnn_sha256=report['cnn_sha256'], layer_name='svm_features',
            feature_dim=512,
            preprocessing_version='rgb-v4-inference-float32-div255',
        )
    os.replace(temp_path, RGB_FEATURE_PATH)
    print('Ekstraksi selesai (detik):', round(time.perf_counter() - started, 1))

assert X_A.shape == X_B.shape == (50000, 512)
assert np.isfinite(X_A).all() and np.isfinite(X_B).all()
FEATURE_ARCHIVE_SHA = sha256_file(RGB_FEATURE_PATH)
features = X_B if SVM_REPRESENTATION == 'B' else X_A
print('Fitur SVM:', features.shape, '| SHA arsip:', FEATURE_ARCHIVE_SHA[:16])
del X_A, X_B
gc.collect()

Ekstraksi RGB: 6400/50000
Ekstraksi RGB: 12800/50000
Ekstraksi RGB: 19200/50000
Ekstraksi RGB: 25600/50000
Ekstraksi RGB: 32000/50000
Ekstraksi RGB: 38400/50000
Ekstraksi RGB: 44800/50000
Ekstraksi RGB: 50000/50000
Ekstraksi selesai (detik): 68.2
Fitur SVM: (50000, 512) | SHA arsip: 452fb86d2742166b


0

## 9. GridSearchCV SVM pada subset development

Agar tidak mengulang training CNN, pencarian hanya melatih SVM. `GridSearchCV` memakai **12.000 training** dan **3.000 validation** yang diambil secara stratified dari split 40K/10K. `PredefinedSplit` menjaga seluruh validation tetap di luar `.fit()` untuk setiap kandidat. `StandardScaler` dan `Normalizer` merupakan bagian dari `Pipeline` agar tidak terjadi data leakage. Pencarian awal menggunakan enam kombinasi `C` dan `gamma`; hasilnya dipakai untuk fitting SVM development pada seluruh 40K training. Proses ini berjalan terutama di CPU, bukan GPU.

In [9]:
screen_train, _ = train_test_split(
    idx_train, train_size=12000, random_state=SEED,
    stratify=y_all[idx_train],
)
screen_val, _ = train_test_split(
    idx_val, train_size=3000, random_state=SEED,
    stratify=y_all[idx_val],
)
search_indices = np.concatenate([screen_train, screen_val])
fold = np.concatenate([
    np.full(len(screen_train), -1, dtype=np.int8),
    np.zeros(len(screen_val), dtype=np.int8),
])
assert len(np.intersect1d(screen_train, screen_val)) == 0


def new_rgb_svm_pipeline(**params):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('normalizer', Normalizer(norm='l2')),
        ('svm', SVC(kernel='rbf', cache_size=1024, **params)),
    ])

if GRID_REPORT_PATH.exists():
    grid_report = json.loads(GRID_REPORT_PATH.read_text(encoding='utf-8'))
    if (grid_report.get('run_id') != RUN_ID
        or grid_report.get('feature_archive_sha256') != FEATURE_ARCHIVE_SHA
        or grid_report.get('split_hash') != split_hash
        or grid_report.get('feature_representation') != SVM_REPRESENTATION
        or grid_report.get('param_grid') != SVM_PARAM_GRID):
        raise ValueError('Hasil grid lama tidak cocok dengan data/konfigurasi. Gunakan run baru.')
    best_params = grid_report['best_params']
    print('Menggunakan hasil GridSearchCV yang tersedia:', best_params)
else:
    search = GridSearchCV(
        new_rgb_svm_pipeline(), param_grid=SVM_PARAM_GRID,
        cv=PredefinedSplit(fold), scoring='accuracy',
        refit=False, n_jobs=1, verbose=2, error_score='raise',
    )
    search_start = time.perf_counter()
    search.fit(features[search_indices], y_all[search_indices])
    search_seconds = time.perf_counter() - search_start
    best_params = dict(search.best_params_)
    grid_report = {
        'run_id': RUN_ID, 'feature_archive_sha256': FEATURE_ARCHIVE_SHA,
        'cnn_sha256': report['cnn_sha256'], 'split_hash': split_hash,
        'feature_representation': SVM_REPRESENTATION,
        'n_screen_train': 12000, 'n_screen_validation': 3000,
        'param_grid': SVM_PARAM_GRID,
        'best_params': best_params, 'best_accuracy': float(search.best_score_),
        'seconds': float(search_seconds),
        'candidate_accuracy': [float(v) for v in search.cv_results_['mean_test_score']],
        'candidate_params': [dict(v) for v in search.cv_results_['params']],
    }
    tmp_grid = GRID_REPORT_PATH.with_suffix('.json.tmp')
    tmp_grid.write_text(json.dumps(grid_report, indent=2, ensure_ascii=False), encoding='utf-8')
    os.replace(tmp_grid, GRID_REPORT_PATH)
    print('GridSearchCV selesai (menit):', round(search_seconds / 60, 1))
print('Hyperparameter terpilih:', best_params)
print('Accuracy screening:', f"{grid_report['best_accuracy']:.2%}")

Fitting 1 folds for each of 6 candidates, totalling 6 fits
[CV] END .......................svm__C=1.0, svm__gamma=scale; total time=   1.6s
[CV] END .........................svm__C=1.0, svm__gamma=0.5; total time=   1.4s
[CV] END .........................svm__C=1.0, svm__gamma=1.0; total time=   1.5s
[CV] END ......................svm__C=10.0, svm__gamma=scale; total time=   1.3s
[CV] END ........................svm__C=10.0, svm__gamma=0.5; total time=   0.9s
[CV] END ........................svm__C=10.0, svm__gamma=1.0; total time=   1.3s
GridSearchCV selesai (menit): 0.1
Hyperparameter terpilih: {'svm__C': 1.0, 'svm__gamma': 0.5}
Accuracy screening: 93.10%


## 10. Evaluasi SVM pada seluruh development split 40K/10K

SVM dipasang ulang menggunakan hyperparameter dari GridSearchCV dan **40.000 training samples**. Scaler di-fit hanya pada 40K tersebut; seluruh 10K validation digunakan untuk mengukur hasil. Model ini masih merupakan *development model*, bukan final model untuk official test. Hasil dan model disimpan terpisah sehingga evaluasi dapat diaudit tanpa mengulangi fitting.

In [10]:
DEVELOPMENT_MODEL_PATH = RGB_OUT / 'model' / 'svm_development_40k.joblib'
if DEVELOPMENT_REPORT_PATH.exists() and DEVELOPMENT_MODEL_PATH.exists():
    dev_report = json.loads(DEVELOPMENT_REPORT_PATH.read_text(encoding='utf-8'))
    if (dev_report.get('feature_archive_sha256') != FEATURE_ARCHIVE_SHA
        or dev_report.get('split_hash') != split_hash
        or dev_report.get('best_params') != best_params
        or dev_report.get('cnn_sha256') != report['cnn_sha256']):
        raise ValueError('Laporan SVM development lama tidak sesuai')
    print('Model development sebelumnya sudah ada:',
          f"{dev_report['validation_accuracy']:.2%}")
elif DEVELOPMENT_REPORT_PATH.exists() or DEVELOPMENT_MODEL_PATH.exists():
    raise FileExistsError('Artefak development hanya sebagian ada; tidak menimpanya')
else:
    dev_pipeline = new_rgb_svm_pipeline(
        C=best_params['svm__C'], gamma=best_params['svm__gamma'],
    )
    fit_start = time.perf_counter()
    dev_pipeline.fit(features[idx_train], y_all[idx_train])
    dev_pred = dev_pipeline.predict(features[idx_val])
    fit_seconds = time.perf_counter() - fit_start
    dev_accuracy = float(accuracy_score(y_all[idx_val], dev_pred))
    dev_report = {
        'run_id': RUN_ID, 'classifier': 'RGB CNN features + SVM',
        'stage': 'development_40k_10k',
        'cnn_sha256': report['cnn_sha256'], 'split_hash': split_hash,
        'feature_archive_sha256': FEATURE_ARCHIVE_SHA,
        'feature_representation': SVM_REPRESENTATION,
        'best_params': best_params,
        'train_count': 40000, 'validation_count': 10000,
        'validation_accuracy': dev_accuracy,
        'validation_macro_f1': float(f1_score(y_all[idx_val], dev_pred, average='macro')),
        'seconds': float(fit_seconds),
    }
    tmp_dev = DEVELOPMENT_MODEL_PATH.with_suffix('.joblib.tmp')
    joblib.dump(dev_pipeline, tmp_dev)
    os.replace(tmp_dev, DEVELOPMENT_MODEL_PATH)
    tmp_report = DEVELOPMENT_REPORT_PATH.with_suffix('.json.tmp')
    tmp_report.write_text(json.dumps(dev_report, indent=2, ensure_ascii=False), encoding='utf-8')
    os.replace(tmp_report, DEVELOPMENT_REPORT_PATH)
    del dev_pipeline
    print('SVM development selesai (menit):', round(fit_seconds / 60, 1))
print('Accuracy validation RGB CNN + SVM:', f"{dev_report['validation_accuracy']:.2%}")

SVM development selesai (menit): 0.1
Accuracy validation RGB CNN + SVM: 93.49%


## 11. Fit final SVM RGB pada seluruh 50.000 training samples

Setelah hyperparameter dipilih menggunakan development validation, `Pipeline` baru dilatih pada **seluruh 50.000 official training features**. Ini tetap training **SVM saja**, bukan CNN. `StandardScaler` dipelajari ulang pada 50K training. Model final disimpan di `model/svm.joblib` dengan metadata CNN, hash feature archive dan identitas split. Set `RUN_FINAL_SVM_50K=False` hanya jika ingin menunda fitting final; dalam kondisi tersebut Testing SVM tidak boleh dijalankan.

In [11]:
if not RUN_FINAL_SVM_50K:
    print('Fitting final SVM RGB 50K dilewati berdasarkan konfigurasi.')
elif SVM_ARTIFACT_PATH.exists() and SVM_REPORT_PATH.exists():
    final_svm_report = json.loads(SVM_REPORT_PATH.read_text(encoding='utf-8'))
    if (final_svm_report.get('cnn_sha256') != report['cnn_sha256']
        or final_svm_report.get('feature_archive_sha256') != FEATURE_ARCHIVE_SHA
        or final_svm_report.get('split_hash') != split_hash
        or final_svm_report.get('best_params') != best_params
        or final_svm_report.get('fit_sample_count') != 50000):
        raise ValueError('Model final SVM yang tersimpan tidak cocok dengan konfigurasi')
    print('Model SVM final 50K sudah ada, tidak dilakukan fitting ulang:', SVM_ARTIFACT_PATH)
elif SVM_ARTIFACT_PATH.exists() or SVM_REPORT_PATH.exists():
    raise FileExistsError('Artefak SVM final hanya sebagian tersedia; tidak menimpanya')
else:
    final_svm = new_rgb_svm_pipeline(
        C=best_params['svm__C'], gamma=best_params['svm__gamma'],
    )
    fit_start = time.perf_counter()
    final_svm.fit(features, y_all)
    fit_seconds = time.perf_counter() - fit_start
    payload = {
        'pipeline': final_svm,
        'run_id': RUN_ID,
        'classifier': 'RGB CNN features + SVM',
        'cnn_sha256': report['cnn_sha256'],
        'feature_archive_sha256': FEATURE_ARCHIVE_SHA,
        'feature_layer': 'svm_features',
        'feature_representation': SVM_REPRESENTATION,
        'feature_dim': 512,
        'split_hash': split_hash,
        'fit_sample_count': 50000,
        'best_params': best_params,
    }
    tmp_model = SVM_ARTIFACT_PATH.with_suffix('.joblib.tmp')
    joblib.dump(payload, tmp_model)
    os.replace(tmp_model, SVM_ARTIFACT_PATH)
    final_svm_report = {
        'run_id': RUN_ID, 'stage': 'final_50k',
        'classifier': 'RGB CNN features + SVM',
        'cnn_sha256': report['cnn_sha256'],
        'feature_archive_sha256': FEATURE_ARCHIVE_SHA,
        'split_hash': split_hash,
        'feature_layer': 'svm_features',
        'feature_dim': 512,
        'feature_representation': SVM_REPRESENTATION,
        'best_params': best_params,
        'fit_sample_count': 50000,
        'fit_seconds': float(fit_seconds),
        'development_validation_accuracy': float(dev_report['validation_accuracy']),
    }
    tmp_report = SVM_REPORT_PATH.with_suffix('.json.tmp')
    tmp_report.write_text(json.dumps(final_svm_report, indent=2, ensure_ascii=False), encoding='utf-8')
    os.replace(tmp_report, SVM_REPORT_PATH)
    del final_svm
    print('Fitting final SVM 50K selesai (menit):', round(fit_seconds / 60, 1))
if RUN_FINAL_SVM_50K:
    print('Model SVM RGB final:', SVM_ARTIFACT_PATH)

Fitting final SVM 50K selesai (menit): 0.2
Model SVM RGB final: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_rgb_residual_v4/rgb/model/svm.joblib


## 12. Ringkasan CNN dan SVM RGB v4

Ringkasan membedakan akurasi validation **CNN Softmax** dan **RGB CNN + SVM** agar keduanya tidak tertukar. Evaluasi pada official test dilakukan dalam **RGB Testing v4**, bukan notebook ini. Jika tahap ekstraksi sudah menyimpan arsip A/B, **RGB Extraction v4 tidak perlu dijalankan lagi**; Feature Engineering Combination v4 dapat langsung memakai file tersebut.

In [12]:
print('=' * 68)
print('HASIL DEVELOPMENT RGB v4')
print('=' * 68)
print('CNN Softmax validation:', f"{report['best_cnn_validation_accuracy']:.2%}")
print('CNN feature + SVM validation:', f"{dev_report['validation_accuracy']:.2%}")
print('Fitur fusion A/B:', RGB_FEATURE_PATH)
print('Checkpoint CNN:', final_path)
print('SVM final 50K:', SVM_ARTIFACT_PATH if RUN_FINAL_SVM_50K else 'BELUM DILATIH')
print('Jalankan Combination v4 dan Training Fusion v4 sebelum official test.')

HASIL DEVELOPMENT RGB v4
CNN Softmax validation: 92.96%
CNN feature + SVM validation: 93.49%
Fitur fusion A/B: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_rgb_residual_v4/rgb/fitur/train_features.npz
Checkpoint CNN: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_rgb_residual_v4/rgb/model/cnn.keras
SVM final 50K: /home/saga/anaconda3/envs/envGPU/BDA_Project/tugas-after-uts/outputs/cifar10_rgb_residual_v4/rgb/model/svm.joblib
Jalankan Combination v4 dan Training Fusion v4 sebelum official test.
